In [16]:
import os
import fitz  
import requests
from typing import Annotated, List
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat, SelectorGroupChat
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from docx import Document as WordDocument
import json
import re
import time
from dotenv import load_dotenv
import asyncio
import aiohttp
from pydantic import BaseModel
from urllib.parse import urlparse
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import os
import time



from openai import OpenAI
import json

client = OpenAI()

load_dotenv()

# Load environment variables
openai_api_key= os.getenv('OPENAI_API_KEY')
LINKUP_API_KEY = os.getenv("LINKUP_API_KEY")

        
# Model Configurations
model_client = OpenAIChatCompletionClient(
    model = 'gpt-4o-mini',
    api_key=openai_api_key
    )

In [17]:
# embedding the dataset in the vector database and creating a retriever for it
import pandas as pd

df = pd.read_csv("Scrapped Gov Schemes Dataset.csv")

# Drop empty column
df = df.drop(columns=["Unnamed: 9"], errors="ignore")

# Fill NaN values
df = df.fillna("")

print(df.columns)

Index(['scheme_name', 'slug', 'details', 'benefits', 'eligibility',
       'application', 'documents', 'level', 'schemeCategory', 'tags'],
      dtype='str')


In [ ]:
documents = []

for _, row in df.iterrows():

    content = f"""
    Scheme Name: {row['scheme_name']}
    Category: {row['schemeCategory']}
    Level: {row['level']}
    
    Description:
    {row['details']}
    
    Eligibility:
    {row['eligibility']}
    
    Benefits:
    {row['benefits']}
    """

    documents.append(
        Document(
            page_content=content,
            metadata={
                "scheme_name": row["scheme_name"],
                "slug": row["slug"],
                "category": row["schemeCategory"],
                "level": row["level"]
            }
        )
    )
    
print(f"Total documents created: {len(documents)}")

Total documents created: 3400


In [19]:


embeddings = OpenAIEmbeddings(model="text-embedding-3-small",chunk_size=50)

vectorstore = Chroma.from_documents(
    documents,
    embedding=embeddings,   
    persist_directory="scheme_chroma_db"
)

vectorstore.persist()

C:\Users\gaikw\AppData\Local\Temp\ipykernel_15952\3197459870.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [28]:
def get_user_details():
   print("Please provide the following details:\n")
   
   user_data = {}
   
   # Basic Details
   user_data["preferred_language"] = input("Preferred Language (e.g., Hindi, English, Marathi): ").strip()
   user_data["name"] = input("Full Name: ").strip()
   user_data["age"] = input("Age: ").strip()
   user_data["gender"] = input("Gender (Male/Female/Other): ").strip()
   user_data["marital_status"] = input("Marital Status (Single/Married/Other): ").strip()
   user_data["caste"] = input("Caste (Optional, press Enter to skip): ").strip()
   # Location Details
   print("\n--- Location Details ---")
   user_data["city"] = input("City/Town/Village: ").strip()
   user_data["state"] = input("State: ").strip()
   user_data["area_type"] = input("Area Type (Urban/Rural): ").strip()
   # Education and Employment
   print("\n--- Education & Employment ---")
   user_data["education"] = input("Highest Education Level: ").strip()
   user_data["employment"] = input("Employment Type (Student/Unemployed/Self-employed/Private/Government): ").strip()
   user_data["income"] = input("Monthly Income (in ₹ or 'N/A'): ").strip()
   # Interest and Query
   print("\n--- Additional Information ---")
   user_data["interest_sector"] = input("Interest Sector (e.g., Agriculture, Business, Education, Health): ").strip()
   user_data["user_query"] = input(f"Please enter your query in your preferred language ({user_data['preferred_language']}): ").strip()
   print("\n✅ All details collected successfully!\n")

   return user_data


In [21]:
def build_enriched_query(user_data):
    
    enriched_query = f"""
    User Profile Information:
    
    Name: {user_data.get('name')}
    Age: {user_data.get('age')}
    Gender: {user_data.get('gender')}
    Marital Status: {user_data.get('marital_status')}
    Caste: {user_data.get('caste')}
    
    Location:
    City: {user_data.get('city')}
    State: {user_data.get('state')}
    Area Type: {user_data.get('area_type')}
    
    Education & Employment:
    Education Level: {user_data.get('education')}
    Employment Type: {user_data.get('employment')}
    Monthly Income: {user_data.get('income')}
    
    Sector of Interest:
    {user_data.get('interest_sector')}
    
    User Query:
    {user_data.get('user_query')}
    
    Based on this profile, suggest government schemes where this person is eligible.
    """
    
    return enriched_query


In [22]:
def recommend_schemes(user_data, k=6):
    
    enriched_query = build_enriched_query(user_data)
    
    results = vectorstore.max_marginal_relevance_search(
        enriched_query,
        k=k,
        fetch_k=20
    )
    
    # Extract slugs
    recommended_slugs = [doc.metadata["slug"] for doc in results]
    
    return recommended_slugs


In [23]:
def get_complete_scheme_details(slugs, df):
    
    complete_info = df[df["slug"].isin(slugs)]
    
    return complete_info

def prepare_llm_context(df_filtered):
    
    scheme_blocks = []
    
    for _, row in df_filtered.iterrows():
        
        scheme_text = f"""
        Scheme Name: {row['scheme_name']}
        Category: {row['schemeCategory']}
        Level: {row['level']}
        
        Description:
        {row['details']}
        
        Benefits:
        {row['benefits']}
        
        Eligibility:
        {row['eligibility']}
        
        Application Process:
        {row['application']}
        
        Required Documents:
        {row['documents']}
        
        Tags:
        {row['tags']}
        """
        
        scheme_blocks.append(scheme_text)
    
    return "\n\n".join(scheme_blocks)


In [29]:
def generate_structured_json(user_data, scheme_context):
    
    prompt = f"""
    You are an AI Government Scheme Advisor.
    
    User Profile:
    {json.dumps(user_data, indent=2)}
    
    Below are the retrieved government schemes:
    
    {scheme_context}
    
    Your task:
    1. Select and rank the TOP 3 most relevant schemes for this user.
    2. Do NOT include more or fewer than 3 schemes.
    3. Rank them from most relevant (1) to least relevant (3).
    4. Do NOT add information not present in the scheme data.
    5. Return ONLY valid JSON.
    
    Required JSON Format:
    
    {{
      "user_profile_summary": "...",
      "recommended_schemes": [
        {{
          "rank": 1,
          "scheme_name": "...",
          "relevance_reason": "...",
          "benefits": "...",
          "eligibility_summary": "...",
          "application_process": "...",
          "required_documents": "...",
          "level": "...",
          "category": "..."
        }},
        {{
          "rank": 2,
          "scheme_name": "...",
          "relevance_reason": "...",
          "benefits": "...",
          "eligibility_summary": "...",
          "application_process": "...",
          "required_documents": "...",
          "level": "...",
          "category": "..."
        }},
        {{
          "rank": 3,
          "scheme_name": "...",
          "relevance_reason": "...",
          "benefits": "...",
          "eligibility_summary": "...",
          "application_process": "...",
          "required_documents": "...",
          "level": "...",
          "category": "..."
        }}
      ]
    }}
    """
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",   # better + cheaper
        temperature=0,
        messages=[
            {"role": "system", "content": "Return strictly valid JSON with exactly 3 recommended schemes."},
            {"role": "user", "content": prompt}
        ]
    )
    
    output_text = response.choices[0].message.content
    
    return json.loads(output_text)

In [ ]:
async def main():

   user_data = get_user_details()

   agent_1_prompt = f"""
   You are a multilingual translation agent specialized in converting user queries into fluent English.
   
   ### Objective:
   Translate the user's query from their known preferred language into clear, natural English while fully preserving its meaning and context.
   
   ### Inputs:
   - user_preferred_language: {user_data['preferred_language']}
   - user_query: {user_data['user_query']}
   
   ### Instructions:
   - Translate the query **only** from the specified user_preferred_language to English.
   - Maintain tone, politeness, and intent (e.g., question, request, or statement).
   - Avoid literal word-by-word translation — ensure it sounds natural.
   - Do not add any explanations or commentary.
   
   ### Output Format:
   {{
     "english_query": "translated query in English"
   }}
   """
   agent_1= AssistantAgent(name="eng_translator", model_client=model_client, system_message=agent_1_prompt)
   
   max_messages_termination = MaxMessageTermination(2)
   team = RoundRobinGroupChat(
       participants=[agent_1],
       termination_condition=max_messages_termination
   )
   result = await team.run(task="Convert the user query from preferred language to English")   
   
   translated_query =  result.messages[-1].content

   
   agent_2_prompt = f"""
   You are a multilingual translation and localization expert.
   
   ### Objective:
   Translate the provided English response into the user's original detected language while keeping it:
   - Clear and easy to understand for common citizens.
   - Grammatically correct and naturally flowing in that language.
   - Faithful to the meaning of the original English text (do not summarize or alter facts).
   
   ### Input:
   - user_prefered_language: {user_data['preferred_language']}
   - english_response: <search_agent_output>
   
   ### Output Format:
   {{
     "translated_response": "<final response translated in user's language>"
   }}
   
   If the prefered_language is 'English', simply return the english_response as is.
   """
   

       
   # Stage 1 – Retrieve Slugs
   slugs = recommend_schemes(user_data)
   
   # Stage 2 – Fetch Full Info
   full_info = get_complete_scheme_details(slugs, df)
   
   
   # Stage 3 – Prepare Context
   scheme_context = prepare_llm_context(full_info)
   print (scheme_context)
   # Stage 4 – Generate Structured JSON
   final_json = generate_structured_json(user_data, scheme_context)
   
   print(json.dumps(final_json, indent=2))

   
   native_translator= AssistantAgent(name="Native_Translator", model_client=model_client, system_message=agent_2_prompt)

   max_messages_termination = MaxMessageTermination(2)
   team2= RoundRobinGroupChat(
       participants=[native_translator],
       termination_condition=max_messages_termination
   )

   result2 = await team2.run(task=f"""Translate this into {user_data['preferred_language']}:\n{json.dumps(final_json, indent=2)}
   as it is and keep the structured response same""")
   final_output = result2.messages[-1].content
   with open("final_output.json", "w", encoding="utf-8") as f:
    json.dump(final_output, f, ensure_ascii=False, indent=2)

   print("✅ Final output saved to final_output.json")
   print(final_output)

In [32]:
await main()

Please provide the following details:


--- Location Details ---

--- Education & Employment ---

--- Additional Information ---

✅ All details collected successfully!

{
  "user_profile_summary": "Shreya is a 22-year-old single female from a rural area in Maharashtra, self-employed with an interest in health and an income of \u20b920,000.",
  "recommended_schemes": [
    {
      "rank": 1,
      "scheme_name": "Mamta Taruni Abhiyan",
      "relevance_reason": "This scheme focuses on health and wellness for women and adolescent girls, which aligns with Shreya's interest in health.",
      "benefits": "Access to health check-ups, antenatal care, postnatal care, nutritional support, health education, and financial support for pregnant women.",
      "eligibility_summary": "Open to adolescent girls aged 10-19 and pregnant women who are residents of Gujarat, with priority for BPL families.",
      "application_process": "Visit the nearest healthcare center, fill the application form, submi

In [27]:
import sys
print(sys.executable)

e:\BE project\scheme_env\Scripts\python.exe
